In [ ]:
#====================================================================
# ECC code structure revised - added Guassian pyramids for robustness
#====================================================================

import cv2
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from pathlib import Path
import os

# === Settings ===
input_folder = "input folder"   # <--- update
output_folder = "output folder" # <--- update
Path(output_folder).mkdir(exist_ok=True)

gaussian_blur_kernel = (5, 5)
pyramid_levels = 3   # number of pyramid levels (coarse-to-fine)

# --- Multilevel ECC Alignment ---
def align_image_ecc_pyramid(ref_img, target_img, levels=3):
    ref = ref_img.astype(np.float32) / 255.0
    tgt = target_img.astype(np.float32) / 255.0

    # build pyramids (coarse -> fine)
    ref_pyr = [ref]
    tgt_pyr = [tgt]
    for _ in range(1, levels):
        ref_pyr.insert(0, cv2.pyrDown(ref_pyr[0]))
        tgt_pyr.insert(0, cv2.pyrDown(tgt_pyr[0]))

    # start with identity warp
    warp_matrix = np.eye(2, 3, dtype=np.float32)

    criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 500, 1e-6)

    # go from coarse to fine
    for lvl in range(levels):
        ref_lvl, tgt_lvl = ref_pyr[lvl], tgt_pyr[lvl]

        try:
            cc, warp_matrix = cv2.findTransformECC(
                ref_lvl, tgt_lvl, warp_matrix,
                motionType=cv2.MOTION_AFFINE,
                criteria=criteria
            )
        except cv2.error as e:
            print(f"ECC failed at level {lvl}: {e}")
            return None

        # scale translation for next finer level
        if lvl < levels - 1:
            warp_matrix[0, 2] *= 2.0
            warp_matrix[1, 2] *= 2.0

    # warp full-resolution image
    aligned = cv2.warpAffine(
        target_img, warp_matrix,
        (ref_img.shape[1], ref_img.shape[0]),
        flags=cv2.INTER_LINEAR + cv2.WARP_INVERSE_MAP
    )
    return aligned

# --- Heatmap plotting ---
def plot_heatmap(diff_img, img_name, output_path):
    plt.figure(figsize=(10, 6))
    im = plt.imshow(diff_img, cmap='seismic', vmin=-256, vmax=256)
    plt.colorbar(im, label="Pixel Intensity Difference")
    plt.title(f"Difference Heatmap\n{img_name}")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()

# --- Main loop ---
image_paths = sorted(glob(os.path.join(input_folder, "*.bmp"))) #<-------- UPDATE FILE TYPE - WILL NOT WORK WITH WRONG FILE TYPE (EX: .jpg, .tif, .png, etc.)
assert len(image_paths) >= 2, "Need at least two images!"

ref_img = cv2.imread(image_paths[0], cv2.IMREAD_GRAYSCALE)
ref_img = cv2.GaussianBlur(ref_img, gaussian_blur_kernel, 0)

for i in range(1, len(image_paths)):
    tgt_path = image_paths[i]
    tgt_img = cv2.imread(tgt_path, cv2.IMREAD_GRAYSCALE)
    tgt_img = cv2.GaussianBlur(tgt_img, gaussian_blur_kernel, 0)

    img_name = f"{Path(tgt_path).stem} - {Path(image_paths[0]).stem}"

    aligned = align_image_ecc_pyramid(ref_img, tgt_img, pyramid_levels)

    if aligned is None:
        print(f"Skipping {img_name} due to ECC failure.")
        continue

    # signed pixel difference
    diff_img = aligned.astype(np.float32) - ref_img.astype(np.float32)

    # save heatmap
    heatmap_path = f"{output_folder}/heatmap_{img_name}.png"
    plot_heatmap(diff_img, img_name, heatmap_path)

    print(f"Processed: {img_name}")

print("ECC batch alignment complete!")
